# 03 K and Distance Weights

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`EXP005_KNN_Normal_K_Weights_fixed.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP005_KNN_Normal_K_Weights_fixed.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)
- [`EXP010_KNN_Normal_Distance_Metric.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)
- [`EXP010_KNN_Normal_Distance_Metric_fast.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric_fast.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)
- [`EXP015_KNN_Normal_K_Weights_HOG.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP015_KNN_Normal_K_Weights_HOG.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/KNN/Normal/scripts/EXP005_KNN_Normal_K_Weights_fixed.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '226218261918b27a8139644108ab8836c718f0ca'}, '05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '17d8d684304198426d4505d25b0d259b373871b9'}, '05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric_fast.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '64b12302c6337471be5103a45b150a7fcf5b1262'}, '05_model_optimize/KNN/Normal/scripts/EXP015_KNN_Normal_K_Weights_HOG.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '05a013ad926281544f1e6d415d24110d58cc92ed'}}

## EXP005_KNN_Normal_K_Weights_fixed

원본: [05_model_optimize/KNN/Normal/scripts/EXP005_KNN_Normal_K_Weights_fixed.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP005_KNN_Normal_K_Weights_fixed.py)  
Git blob: `226218261918b27a8139644108ab8836c718f0ca`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-005: CoM KNN Normal neighbor count and vote weights comparison.

Expected repository files:
- 03_experiments/model_optimization/common.py
- 05_model_optimize/MLP/Normal/scripts/Automatic_Centering_Transformer.py
- 05_model_optimize/KNN/Normal/scripts/EXP004_KNN_Normal_CoM.py

Train on the EXACT official training fold. Evaluate official Validation and
four dev Hand-made writers, never use official Original Test / final Test.
The 14 candidates reuse one 15-NN query. This is a development exper



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/EXP005_KNN_Normal_K_Weights_fixed.py
#!/usr/bin/env python3
"""EXP-005: CoM KNN Normal neighbor count and vote weights comparison.

Expected repository files:
- 03_experiments/model_optimization/common.py
- 05_model_optimize/MLP/Normal/scripts/Automatic_Centering_Transformer.py
- 05_model_optimize/KNN/Normal/scripts/EXP004_KNN_Normal_CoM.py

Train on the EXACT official training fold. Evaluate official Validation and
four dev Hand-made writers, never use official Original Test / final Test.
The 14 candidates reuse one 15-NN query. This is a development experiment,
not an independent final generalization estimate.
"""
from __future__ import annotations

import csv
import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import sklearn
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

ROOT = Path(__file__).resolve().parents[4]
sys.path.insert(0, str(ROOT / '03_experiments' / 'model_optimization'))
sys.path.insert(0, str(ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'))

from common import LABELS, fixed_split, pipeline, scores
from EXP004_KNN_Normal_CoM import BatchedCoM, read_handmade, test_centering

OUT = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP005_k_weights'
K_VALUES = (1, 3, 5, 7, 9, 11, 15)
WEIGHTS = ('uniform', 'distance')
MIN_OFFICIAL_ACC = 0.9569791666666667  # CoM baseline 0.961979 - 0.005 (0.5%p)
BASE_OFFICIAL_ACC = 0.9619791666666667
BASE_OFFICIAL_F1 = 0.9619584413989763
BASE_HANDMADE_ACC = 0.778515625
BASE_HANDMADE_F1 = 0.7764856058056413


def csv_write(path: Path, rows: list[dict], columns=None):
    columns = columns or list(rows[0])
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=columns)
        w.writeheader()
        w.writerows(rows)


def predict_from_neighbors(indices, distances, training_labels, k: int, weights: str):
    labels = training_labels[indices[:, :k]]
    d = distances[:, :k]
    if weights == 'uniform':
        w = np.ones(d.shape, dtype=np.float64)
    else:
        zeros = d == 0
        w = np.zeros(d.shape, dtype=np.float64)
        np.divide(1.0, d, out=w, where=~zeros)
        w = np.where(zeros.any(axis=1, keepdims=True), zeros.astype(np.float64), w)
    votes = np.empty((len(labels), len(LABELS)), dtype=np.float64)
    for c in range(len(LABELS)):
        votes[:, c] = np.sum((labels == c) * w, axis=1)
    return np.argmax(votes, axis=1).astype(np.int64)


def transform_before_knn(fitted_model, X):
    """Apply already fitted preprocessing steps without slicing a Pipeline.

    A Pipeline slice ending in a stateless CoM/scaler can fail the
    scikit-learn fitted-state check despite the full pipeline being fitted.
    """
    transformed = X
    for name, transformer in fitted_model.steps[:-1]:
        transformed = transformer.transform(transformed)
    return transformed


def paired_counts(y, old, new):
    a, b = old == y, new == y
    return {'corrected': int(np.sum(~a & b)),
            'new_errors': int(np.sum(a & ~b)),
            'unchanged_correct': int(np.sum(a & b)),
            'unchanged_wrong': int(np.sum(~a & ~b))}


def main():
    test_centering()
    xtr, ytr, xva, yva, dataset_sha = fixed_split()
    xhm, yhm, writer_ids, writer_indices = read_handmade()
    print(f'Official Training/Validation: {len(ytr)}/{len(yva)}; Hand-made: {len(yhm)}', flush=True)

    knn = KNeighborsClassifier(n_neighbors=max(K_VALUES), weights='uniform', metric='minkowski', p=2)
    model = Pipeline([('center_of_mass', BatchedCoM()), *pipeline(knn, True).steps])
    start = time.perf_counter()
    print('Fitting KNN with unchanged CoM on official Training only...', flush=True)
    model.fit(xtr, ytr)
    fit_sec = time.perf_counter() - start
    model_knn = model.named_steps['model']

    print('Computing 15 nearest neighbors on official Validation...', flush=True)
    start = time.perf_counter()
    dva, iva = model_knn.kneighbors(transform_before_knn(model, xva), n_neighbors=max(K_VALUES), return_distance=True)
    query_va_sec = time.perf_counter() - start
    print('Computing 15 nearest neighbors on Hand-made...', flush=True)
    start = time.perf_counter()
    dhm, ihm = model_knn.kneighbors(transform_before_knn(model, xhm), n_neighbors=max(K_VALUES), return_distance=True)
    query_hm_sec = time.perf_counter() - start

    predictions = {}
    rows, writer_rows = [], []
    for k in K_VALUES:
        for weights in WEIGHTS:
            key = f'k{k}_{weights}'
            pred_va = predict_from_neighbors(iva, dva, ytr, k, weights)
            pred_hm = predict_from_neighbors(ihm, dhm, ytr, k, weights)
            eva, ehm = scores(yva, pred_va), scores(yhm, pred_hm)
            predictions[key] = (pred_va, pred_hm)
            row = {'candidate': key, 'k': k, 'weights': weights,
                   'official_accuracy': eva['accuracy'], 'official_macro_f1': eva['macro_f1'],
                   'handmade_accuracy': ehm['accuracy'], 'handmade_macro_f1': ehm['macro_f1'],
                   'official_delta_pp_vs_com_k5': 100 * (eva['accuracy'] - BASE_OFFICIAL_ACC),
                   'handmade_delta_pp_vs_com_k5': 100 * (ehm['accuracy'] - BASE_HANDMADE_ACC),
                   'official_guardrail_pass': int(eva['accuracy'] >= MIN_OFFICIAL_ACC)}
            rows.append(row)
            for w in (41, 42, 43, 44):
                mask = writer_ids == w
                e = scores(yhm[mask], pred_hm[mask])
                writer_rows.append({'candidate': key, 'writer': w, 'n': int(mask.sum()),
                                    'accuracy': e['accuracy'], 'macro_f1': e['macro_f1']})
            print(f"{key:13s} Official={eva['accuracy']:.6f} F1={eva['macro_f1']:.6f} | "
                  f"Hand-made={ehm['accuracy']:.6f} F1={ehm['macro_f1']:.6f} | "
                  f"guardrail={'PASS' if row['official_guardrail_pass'] else 'FAIL'}", flush=True)

    ref_va, ref_hm = predictions['k5_uniform']
    ref_va_s, ref_hm_s = scores(yva, ref_va), scores(yhm, ref_hm)
    if not (np.isclose(ref_va_s['accuracy'], BASE_OFFICIAL_ACC, atol=1e-12)
            and np.isclose(ref_va_s['macro_f1'], BASE_OFFICIAL_F1, atol=1e-12)
            and np.isclose(ref_hm_s['accuracy'], BASE_HANDMADE_ACC, atol=1e-12)
            and np.isclose(ref_hm_s['macro_f1'], BASE_HANDMADE_F1, atol=1e-12)):
        raise RuntimeError('CoM k=5 uniform reference differs from EXP-004. '
                           'Do not compare or record candidate results before investigating '
                           'version/tie handling and source data.')
    print('EXP-004 reference metrics successfully reproduced.', flush=True)

    eligible = [r for r in rows if r['official_guardrail_pass']]
    if not eligible:
        raise RuntimeError('No candidate satisfies official Validation guardrail.')
    chosen = max(eligible, key=lambda r: (r['handmade_macro_f1'], r['handmade_accuracy'],
                                          r['official_macro_f1'], -r['k']))
    best_key = chosen['candidate']
    _, best_hm = predictions[best_key]
    best_hm_scores = scores(yhm, best_hm)
    changes = paired_counts(yhm, ref_hm, best_hm)

    OUT.mkdir(parents=True, exist_ok=True)
    csv_write(OUT / 'candidate_results.csv', rows)
    csv_write(OUT / 'by_writer.csv', writer_rows)
    csv_write(OUT / 'handmade_best_predictions.csv', [
        {'writer': int(w), 'index': int(idx), 'true': LABELS[int(y)],
         'com_k5_prediction': LABELS[int(a)], 'candidate_prediction': LABELS[int(b)],
         'com_k5_correct': int(a == y), 'candidate_correct': int(b == y)}
        for w, idx, y, a, b in zip(writer_ids, writer_indices, yhm, ref_hm, best_hm)
    ])
    cm = best_hm_scores['confusion_matrix']
    csv_write(OUT / 'handmade_best_confusion_matrix.csv', [
        {'true/pred': LABELS[i], **{LABELS[j]: cm[i][j] for j in range(16)}}
        for i in range(16)
    ])
    summary = {'experiment': 'EXP-005 KNN Normal CoM neighbor count and vote weights',
               'official_dataset_sha256': dataset_sha,
               'train_n': len(ytr), 'validation_n': len(yva), 'handmade_n': len(yhm),
               'center_method': 'MLP AutomaticCentering mass, threshold=32, target=(13.5,13.5), integer shift',
               'k_values': list(K_VALUES), 'weights_values': list(WEIGHTS),
               'distance': 'Euclidean (Minkowski p=2)',
               'guardrail': f'official Validation accuracy >= {MIN_OFFICIAL_ACC:.9f}',
               'selection': 'among guardrail pass: highest hand-made macro F1; ties by accuracy, official macro F1, smaller k',
               'note': 'Development selection on Hand-made. These scores are NOT an independent final Test.',
               'reference_com_k5': {'official': ref_va_s, 'handmade': ref_hm_s},
               'candidates': rows, 'chosen': chosen,
               'chosen_vs_reference_handmade_paired': changes,
               'fit_seconds': fit_sec, 'query_official_seconds': query_va_sec,
               'query_handmade_seconds': query_hm_sec,
               'python': platform.python_version(), 'numpy': np.__version__,
               'sklearn': sklearn.__version__}
    (OUT / 'metrics.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'\nSELECTED CANDIDATE (development only): {best_key}', flush=True)
    print(f"Official={chosen['official_accuracy']:.6f}, Hand-made={chosen['handmade_accuracy']:.6f}, "
          f"MacroF1={chosen['handmade_macro_f1']:.6f}", flush=True)
    print('Paired Hand-made vs CoM k5:', changes, flush=True)
    print('Saved:', OUT, flush=True)


if __name__ == '__main__':
    main()


## EXP010_KNN_Normal_Distance_Metric

원본: [05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric.py)  
Git blob: `17d8d684304198426d4505d25b0d259b373871b9`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-010: compare KNN Minkowski distance metrics using EXP-009's exact training setup.

From repository root:
  python 05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric.py

Default: pilot p=1,1.5,2,3 using the SAME stratified pilot train/eval subsets;
then exact full-set p=2 control AND the highest-ranked alternative by pilot
hand-made macro F1. Use --screen-only to skip expensive full evaluations.
Use --full-all to run all four metrics on all validation images (very slow).

CLI 인자 (정적 추출): `--screen-only`, `--full-all`, `--pilot-train-per-class`, `--pilot-official-per-class`, `--pilot-handmade-per-class-per-writer`

In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-010: compare KNN Minkowski distance metrics using EXP-009's exact training setup.

From repository root:
  python 05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric.py

Default: pilot p=1,1.5,2,3 using the SAME stratified pilot train/eval subsets;
then exact full-set p=2 control AND the highest-ranked alternative by pilot
hand-made macro F1. Use --screen-only to skip expensive full evaluations.
Use --full-all to run all four metrics on all validation images (very slow).
"""
from __future__ import annotations

import argparse
import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import sklearn
from sklearn.neighbors import KNeighborsClassifier

ROOT = Path(__file__).resolve().parents[4]
SCRIPTS = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'
sys.path.insert(0, str(ROOT / '03_experiments' / 'model_optimization'))
sys.path.insert(0, str(SCRIPTS))

from common import fixed_split, LABELS, scores
from EXP004_KNN_Normal_CoM import BatchedCoM, read_handmade, paired_stats
from EXP007_KNN_Normal_Augmentation import (
    sample_balanced, size_jitter, stroke_jitter, test_augmentation,
    save_csv, save_cm, OFFICIAL_GUARDRAIL, SEED,
)

OUTPUT = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP010_distance_metrics'
P_VALUES = (1.0, 1.5, 2.0, 3.0)
EXPECTED_FULL_P2 = (18381, 2153)  # EXP-009 0.44, 95.734375% and 84.1015625%


def stratified_indices(y, per_class, seed):
    rng = np.random.default_rng(seed)
    chosen = []
    for lab in range(16):
        candidates = np.flatnonzero(y == lab)
        if len(candidates) < per_class:
            raise ValueError(f'Class {lab} has insufficient samples')
        chosen.extend(rng.choice(candidates, size=per_class, replace=False).tolist())
    return np.asarray(sorted(chosen), dtype=np.int64)


def candidate_name(p):
    return 'p' + str(p).replace('.', '_')


def predict_variant(xtr, ytr, xva, yva, xhm, yhm, p, stage):
    model = KNeighborsClassifier(n_neighbors=3, weights='distance',
                                 metric='minkowski', p=p,
                                 algorithm='brute', n_jobs=1)
    t = time.perf_counter()
    model.fit(xtr, ytr)
    fit_sec = time.perf_counter() - t
    t = time.perf_counter()
    official_pred = model.predict(xva).astype(np.int64)
    off_sec = time.perf_counter() - t
    t = time.perf_counter()
    handmade_pred = model.predict(xhm).astype(np.int64)
    hm_sec = time.perf_counter() - t
    official = scores(yva, official_pred)
    handmade = scores(yhm, handmade_pred)
    rec = {
        'stage': stage, 'variant': candidate_name(p), 'p': p,
        'train_n': len(ytr), 'official_n': len(yva), 'handmade_n': len(yhm),
        'official_correct': int(np.sum(official_pred == yva)),
        'official_accuracy': official['accuracy'],
        'official_macro_f1': official['macro_f1'],
        'handmade_correct': int(np.sum(handmade_pred == yhm)),
        'handmade_accuracy': handmade['accuracy'],
        'handmade_macro_f1': handmade['macro_f1'],
        'official_guardrail_pass': (int(official['accuracy'] >= OFFICIAL_GUARDRAIL)
                                    if stage == 'full' else ''),
        'fit_seconds': fit_sec, 'official_predict_seconds': off_sec,
        'handmade_predict_seconds': hm_sec,
    }
    return rec, official_pred, handmade_pred, official, handmade


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--screen-only', action='store_true',
                        help='Only run stratified pilot on same subsets for all distances')
    parser.add_argument('--full-all', action='store_true',
                        help='Run all metrics on full dataset (computationally expensive)')
    parser.add_argument('--pilot-train-per-class', type=int, default=256,
                        help='Original + augmented pilot samples per class (default 256 total)')
    parser.add_argument('--pilot-official-per-class', type=int, default=32)
    parser.add_argument('--pilot-handmade-per-class-per-writer', type=int, default=4)
    args = parser.parse_args()
    if args.screen_only and args.full_all:
        parser.error('Choose only one of --screen-only and --full-all')
    if not 3 <= args.pilot_train_per_class <= 400:
        parser.error('--pilot-train-per-class must be between 3 and 400')
    if not 1 <= args.pilot_handmade_per_class_per_writer <= 40:
        parser.error('--pilot-handmade-per-class-per-writer must be between 1 and 40')

    test_augmentation()
    xtr, ytr, xva, yva, dataset_sha = fixed_split()
    xhm, yhm, writers, sample_ids = read_handmade()
    if (len(ytr), len(yva), len(yhm)) != (76800, 19200, 2560):
        raise RuntimeError('Dataset split mismatch')
    com = BatchedCoM()
    print('Preparing EXP-009 best: 30% combined augmentation, strength=0.44...', flush=True)
    idx = sample_balanced(ytr, fraction=0.30, seed=SEED)
    sized, size_counts = size_jitter(xtr, idx, seed=SEED)
    augmented, stroke_stats = stroke_jitter(sized, strength=0.44)
    train = np.concatenate((com.transform(xtr), com.transform(augmented)), axis=0)
    train_labels = np.concatenate((ytr, ytr[idx]))
    official = com.transform(xva)
    handmade = com.transform(xhm)
    del sized, augmented, xtr, xva, xhm
    train /= np.float32(255.0)
    official /= np.float32(255.0)
    handmade /= np.float32(255.0)
    if len(train_labels) != 99840:
        raise RuntimeError('Expected 99,840 training rows')
    print('Full train=99,840; official validation=19,200; handmade=2,560', flush=True)

    # Identical, deterministic stratified subsets across all pilot metrics.
    rng = np.random.default_rng(SEED + 100)
    pilot_tr = stratified_indices(train_labels, args.pilot_train_per_class, SEED + 101)
    pilot_va = stratified_indices(yva, args.pilot_official_per_class, SEED + 102)
    pilot_hm = []
    for writer in (41, 42, 43, 44):
        for lab in range(16):
            pool = np.flatnonzero((writers == writer) & (yhm == lab))
            pilot_hm.extend(rng.choice(pool, size=args.pilot_handmade_per_class_per_writer,
                                       replace=False).tolist())
    pilot_hm = np.asarray(sorted(pilot_hm), dtype=np.int64)
    print(f'PILOT ONLY: train={len(pilot_tr)}, official={len(pilot_va)}, handmade={len(pilot_hm)}', flush=True)
    print('Pilot accuracies are NOT comparable with full EXP-009 results.', flush=True)

    OUTPUT.mkdir(parents=True, exist_ok=True)
    pilot_rows = []
    full_rows = []
    full_predictions = {}
    full_detail = {}
    writer_rows = []
    class_rows = []
    handmade_prediction_rows = []
    for p in P_VALUES:
        print(f'\n[PILOT p={p:g}] Starting brute-force distance evaluation...', flush=True)
        result = predict_variant(train[pilot_tr], train_labels[pilot_tr],
                                 official[pilot_va], yva[pilot_va],
                                 handmade[pilot_hm], yhm[pilot_hm], p, 'pilot')
        row = result[0]
        pilot_rows.append(row)
        print(f"{row['variant']}: official={row['official_accuracy']:.6f}, "
              f"handmade={row['handmade_accuracy']:.6f}, "
              f"macroF1={row['handmade_macro_f1']:.6f}; "
              f"predict_s={row['official_predict_seconds'] + row['handmade_predict_seconds']:.1f}", flush=True)
        save_csv(OUTPUT / 'pilot_results.csv', pilot_rows)

    other = max((r for r in pilot_rows if r['p'] != 2.0),
                key=lambda r: (r['handmade_macro_f1'], r['handmade_accuracy'],
                               r['official_macro_f1']))
    full_ps = list(P_VALUES) if args.full_all else [2.0, other['p']]
    print('\nPilot selected alternative:', candidate_name(other['p']), flush=True)
    if args.screen_only:
        full_ps = []
        print('SCREEN-ONLY requested; no full-set comparison / no final model selection.', flush=True)
    else:
        print('FULL evaluation metrics:', ', '.join(map(str, full_ps)), flush=True)
        print('Fractional p metrics on full 99,840-row train can be VERY slow.', flush=True)
    for p in full_ps:
        print(f'\n[FULL p={p:g}] Running on complete validation and handwritten data...', flush=True)
        result = predict_variant(train, train_labels, official, yva,
                                 handmade, yhm, p, 'full')
        row, op, hp, off, hm = result
        if p == 2.0 and (row['official_correct'], row['handmade_correct']) != EXPECTED_FULL_P2:
            raise RuntimeError(f'EXP-009 p=2 control mismatch: '
                               f"{(row['official_correct'], row['handmade_correct'])} "
                               f'!= {EXPECTED_FULL_P2}')
        full_rows.append(row)
        name = candidate_name(p)
        full_predictions[name] = (op, hp)
        full_detail[name] = {'official': off, 'handmade': hm}
        for writer in (41, 42, 43, 44):
            m = writers == writer
            wr = scores(yhm[m], hp[m])
            writer_rows.append({'variant': name, 'writer': writer, 'n': int(m.sum()),
                                'accuracy': wr['accuracy'], 'macro_f1': wr['macro_f1']})
        for lab in LABELS:
            item = hm['per_class'][lab]
            class_rows.append({'variant': name, 'class': lab,
                               'support': item['support'], 'recall': item['recall'],
                               'precision': item['precision'], 'f1': item['f1']})
        for i in range(len(yhm)):
            handmade_prediction_rows.append({'variant': name, 'writer': int(writers[i]),
                                            'index': int(sample_ids[i]), 'true': LABELS[int(yhm[i])],
                                            'predicted': LABELS[int(hp[i])],
                                            'correct': int(hp[i] == yhm[i])})
        save_cm(OUTPUT / f'official_cm_{name}.csv', off['confusion_matrix'])
        save_cm(OUTPUT / f'handmade_cm_{name}.csv', hm['confusion_matrix'])
        save_csv(OUTPUT / 'full_results.csv', full_rows)
        print(f"{name}: official={row['official_accuracy']:.6f} F1={row['official_macro_f1']:.6f}, "
              f"handmade={row['handmade_accuracy']:.6f} F1={row['handmade_macro_f1']:.6f}, "
              f"guardrail={'PASS' if row['official_guardrail_pass'] else 'FAIL'}", flush=True)

    selected = None
    if full_rows:
        valid = [r for r in full_rows if r['official_guardrail_pass']]
        if valid:
            selected = max(valid, key=lambda r: (r['handmade_macro_f1'],
                                                  r['handmade_accuracy'],
                                                  r['official_macro_f1']))
        if writer_rows:
            save_csv(OUTPUT / 'by_writer.csv', writer_rows)
            save_csv(OUTPUT / 'per_class_handmade.csv', class_rows)
            save_csv(OUTPUT / 'handmade_predictions.csv', handmade_prediction_rows)
    paired = {}
    if 'p2_0' in full_predictions:
        baseline_off, baseline_hm = full_predictions['p2_0']
        for name, (op, hp) in full_predictions.items():
            if name != 'p2_0':
                paired[name] = {'official': paired_stats(yva, baseline_off, op),
                                'handmade': paired_stats(yhm, baseline_hm, hp)}
    summary = {
        'experiment': 'EXP-010 KNN Normal Minkowski p comparison',
        'official_dataset_sha256': dataset_sha,
        'train_n': len(train_labels), 'validation_n': len(yva), 'handmade_n': len(yhm),
        'augmentation': {'ratio': 0.30, 'stroke_strength': 0.44,
                         'sizes': size_counts, 'stroke_stats': stroke_stats},
        'knn': {'n_neighbors': 3, 'weights': 'distance', 'metric': 'minkowski',
                'algorithm': 'brute'},
        'pilot': {'train_n': len(pilot_tr), 'official_n': len(pilot_va),
                  'handmade_n': len(pilot_hm), 'rows': pilot_rows,
                  'note': 'Pilot uses a reduced training AND evaluation subset. '
                          'Do not compare pilot accuracy with EXP-009 full-set accuracy.'},
        'full': {'evaluated_p': full_ps, 'rows': full_rows,
                 'selected_development_candidate': selected,
                 'paired_vs_p2': paired},
        'guardrail_full_official_accuracy': OFFICIAL_GUARDRAIL,
        'environment': {'python': platform.python_version(),
                        'numpy': np.__version__, 'sklearn': sklearn.__version__},
        'note': 'Official Training augmentation only. Hand-made used for development '
                'selection only. Held-out original/final Test not accessed.',
    }
    (OUTPUT / 'metrics.json').write_text(json.dumps(summary, indent=2,
                                                   ensure_ascii=False), encoding='utf-8')
    print('\nSELECTED FULL DEVELOPMENT CANDIDATE:',
          selected['variant'] if selected else 'NONE', flush=True)
    print('Saved:', OUTPUT, flush=True)


if __name__ == '__main__':
    main()


## EXP010_KNN_Normal_Distance_Metric_fast

원본: [05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric_fast.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric_fast.py)  
Git blob: `64b12302c6337471be5103a45b150a7fcf5b1262`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-010 fast continuation: reuse completed EXP-010 CSV results, do not rerun distances.

Run from the repository root:
  python 05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric_fast.py

The original EXP-010 writes pilot_results.csv after each pilot and
full_results.csv after every completed full candidate. This script only
reads those files. An interrupted p=1.5 full computation cannot be resumed;
its result is NEVER inferred from its pilot score.



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric_fast.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-010 fast continuation: reuse completed EXP-010 CSV results, do not rerun distances.

Run from the repository root:
  python 05_model_optimize/KNN/Normal/scripts/EXP010_KNN_Normal_Distance_Metric_fast.py

The original EXP-010 writes pilot_results.csv after each pilot and
full_results.csv after every completed full candidate. This script only
reads those files. An interrupted p=1.5 full computation cannot be resumed;
its result is NEVER inferred from its pilot score.
"""
from __future__ import annotations

import csv
import json
import os
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path(os.environ["EXP010_PROJECT_ROOT"]).resolve() if "EXP010_PROJECT_ROOT" in os.environ else Path(__file__).resolve().parents[4]
RESULTS = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP010_distance_metrics'
EXPECTED_P2 = (18381, 2153)
MIN_OFFICIAL_ACC = 0.957


def load_rows(path):
    if not path.exists():
        return []
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))


def numeric(row, name):
    return float(row[name])


def get_p(row):
    return float(row['p'])


def write_csv(path, rows, fields):
    with path.open('w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def main():
    pilot_path = RESULTS / 'pilot_results.csv'
    full_path = RESULTS / 'full_results.csv'
    pilot = load_rows(pilot_path)
    full = load_rows(full_path)
    if not pilot and not full:
        print('No completed EXP-010 CSV found in:', RESULTS)
        print('The original run did not save completed results here. Do not rerun the slow FULL p=1.5 command.')
        return 1
    print(f'Cached results: PILOT={len(pilot)} row(s); FULL={len(full)} row(s)')
    print('NO KNN training or distance computation will be performed.\n')

    unique_pilot = {get_p(row): row for row in pilot}
    unique_full = {get_p(row): row for row in full}
    if 2.0 in unique_full:
        control = unique_full[2.0]
        correct_pair = (int(control['official_correct']), int(control['handmade_correct']))
        if correct_pair != EXPECTED_P2:
            raise RuntimeError(f'EXP-009 baseline mismatch: {correct_pair} != {EXPECTED_P2}')
        print('FULL p=2 EXACT REPRODUCTION VERIFIED.')
    else:
        print('FULL p=2 not present; this continuation cannot verify the EXP-009 baseline.')

    print('\nPILOT (reduced train AND evaluation sets; not full-comparable):')
    pilot_ranked = sorted(unique_pilot.values(), key=lambda r: (
        numeric(r, 'handmade_macro_f1'), numeric(r, 'handmade_accuracy'),
        numeric(r, 'official_macro_f1')), reverse=True)
    for row in sorted(unique_pilot.values(), key=get_p):
        print(f" p={get_p(row):g}: official={numeric(row, 'official_accuracy'):.6f}, "
              f"handmade={numeric(row, 'handmade_accuracy'):.6f}, "
              f"handmade_F1={numeric(row, 'handmade_macro_f1'):.6f}")
    if pilot_ranked:
        print(f" Pilot leader: p={get_p(pilot_ranked[0]):g} (PILOT ONLY)")

    print('\nFULL (only finished, saved models):')
    for row in sorted(unique_full.values(), key=get_p):
        print(f" p={get_p(row):g}: official={numeric(row, 'official_accuracy'):.6f}, "
              f"handmade={numeric(row, 'handmade_accuracy'):.6f}, "
              f"handmade_F1={numeric(row, 'handmade_macro_f1'):.6f}")

    eligible = [r for r in unique_full.values()
                if numeric(r, 'official_accuracy') >= MIN_OFFICIAL_ACC]
    selected = max(eligible, key=lambda r: (
        numeric(r, 'handmade_macro_f1'), numeric(r, 'handmade_accuracy'),
        numeric(r, 'official_macro_f1'))) if eligible else None
    candidate = selected and get_p(selected)
    if candidate is None:
        print('\nNo FULL-evaluated candidate met the official guardrail.')
    else:
        print(f'\nSelected among FULL-evaluated models only: p={candidate:g}')
    if len(unique_full) < 4:
        print('INCOMPLETE SEARCH: other p values were not fully evaluated. No claim of global best p.')

    rows = []
    for p in (1.0, 1.5, 2.0, 3.0):
        a = unique_pilot.get(p)
        b = unique_full.get(p)
        rows.append({
            'p': p,
            'pilot_official_accuracy': a['official_accuracy'] if a else '',
            'pilot_handmade_accuracy': a['handmade_accuracy'] if a else '',
            'pilot_handmade_macro_f1': a['handmade_macro_f1'] if a else '',
            'full_status': 'COMPLETED' if b else 'NOT_COMPLETED',
            'full_official_accuracy': b['official_accuracy'] if b else '',
            'full_handmade_accuracy': b['handmade_accuracy'] if b else '',
            'full_handmade_macro_f1': b['handmade_macro_f1'] if b else '',
            'full_guardrail_pass': (numeric(b, 'official_accuracy') >= MIN_OFFICIAL_ACC) if b else '',
        })
    fields = list(rows[0].keys())
    RESULTS.mkdir(parents=True, exist_ok=True)
    write_csv(RESULTS / 'fast_comparison.csv', rows, fields)
    report = {
        'experiment': 'EXP-010 fast continuation (cached results only)',
        'generated_utc': datetime.now(timezone.utc).isoformat(),
        'source_files': [str(pilot_path), str(full_path)],
        'pilot_rows_count': len(pilot),
        'full_rows_count': len(full),
        'official_guardrail': MIN_OFFICIAL_ACC,
        'full_reproduced_exp009': 2.0 in unique_full,
        'selected_from_completed_full_only_p': candidate,
        'exhaustive_distance_search': len(unique_full) == 4,
        'comparison': rows,
        'limitations': [
            'Pilot has a much smaller training and evaluation set; pilot scores must not be compared with full scores.',
            'An interrupted FULL p=1.5 result is not available and cannot be resumed from partial computation.',
            'Incomplete full search cannot establish that p=2 (or any other p) is globally optimal.',
            'No original final Test data were accessed by this continuation script.',
        ],
    }
    (RESULTS / 'fast_summary.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    with (RESULTS / 'EXP010_fast_summary.md').open('w', encoding='utf-8') as f:
        f.write('# EXP-010 거리 함수 비교 — 중단 후 캐시 결과 정리\n\n')
        f.write('완료된 CSV만 읽었습니다. 추가 KNN 계산은 수행하지 않았습니다.\n\n')
        f.write('| p | PILOT 손글씨 정확도 | FULL 손글씨 정확도 | FULL 공식 정확도 | FULL 상태 |\n')
        f.write('|---:|---:|---:|---:|---|\n')
        for r in rows:
            pil = f"{float(r['pilot_handmade_accuracy']):.2%}" if r['pilot_handmade_accuracy'] != '' else '—'
            hm = f"{float(r['full_handmade_accuracy']):.2%}" if r['full_handmade_accuracy'] != '' else '—'
            off = f"{float(r['full_official_accuracy']):.2%}" if r['full_official_accuracy'] != '' else '—'
            f.write(f"| {r['p']:g} | {pil} | {hm} | {off} | {r['full_status']} |\n")
        f.write('\n**해석:** PILOT과 FULL 성능은 직접 비교할 수 없으며, FULL 미완료 조건에 대한 결론을 내릴 수 없습니다.\n')
        f.write(f'\n**FULL 완료 조건 중 선택:** {f"p={candidate:g}" if candidate is not None else "없음"}\n')
    print('Saved:', RESULTS / 'fast_comparison.csv')
    print('Saved:', RESULTS / 'fast_summary.json')
    print('Saved:', RESULTS / 'EXP010_fast_summary.md')
    return 0


if __name__ == '__main__':
    raise SystemExit(main())


## EXP015_KNN_Normal_K_Weights_HOG

원본: [05_model_optimize/KNN/Normal/scripts/EXP015_KNN_Normal_K_Weights_HOG.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/scripts/EXP015_KNN_Normal_K_Weights_HOG.py)  
Git blob: `05a013ad926281544f1e6d415d24110d58cc92ed`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Normal/logs)

EXP-015: KNN k/weights sweep for the EXACT EXP-014 targeted-10 training set.

Run from repository root:
  python 05_model_optimize/KNN/Normal/scripts/EXP015_KNN_Normal_K_Weights_HOG.py

Only official Training is fitted. Validation and four Hand-made writers are
used for development evaluation. Official Test is never accessed.

Uses the exact EXP014 helper functions/seed/augmentation. A single 11-NN
query is reused across all k/weight variants. Supports k=1,3,5,7,9,11 and
uniform/distance. Result



In [ ]:
%%archive_source 05_model_optimize/KNN/Normal/scripts/EXP015_KNN_Normal_K_Weights_HOG.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-015: KNN k/weights sweep for the EXACT EXP-014 targeted-10 training set.

Run from repository root:
  python 05_model_optimize/KNN/Normal/scripts/EXP015_KNN_Normal_K_Weights_HOG.py

Only official Training is fitted. Validation and four Hand-made writers are
used for development evaluation. Official Test is never accessed.

Uses the exact EXP014 helper functions/seed/augmentation. A single 11-NN
query is reused across all k/weight variants. Supports k=1,3,5,7,9,11 and
uniform/distance. Results are development results, not held-out Test scores.
"""
from __future__ import annotations

import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import sklearn
from sklearn.neighbors import KNeighborsClassifier

ROOT = Path(__file__).resolve().parents[4]
SCRIPTS = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'scripts'
sys.path.insert(0, str(ROOT / '03_experiments' / 'model_optimization'))
sys.path.insert(0, str(SCRIPTS))

from common import LABELS, fixed_split, scores
from EXP004_KNN_Normal_CoM import BatchedCoM, read_handmade, paired_stats, test_centering
from EXP007_KNN_Normal_Augmentation import (
    sample_balanced, size_jitter, stroke_jitter, test_augmentation,
    save_csv, save_cm, OFFICIAL_GUARDRAIL, SEED,
)
from EXP011_KNN_Normal_Deskew import deskew, test_deskew
from EXP012_KNN_Normal_HOG import hog_features, hybrid_features, test_hog
from EXP014_KNN_Normal_Targeted_Augmentation import (
    weak_classes_from_dev_predictions, select_extra, augmented_features,
    PRED_CSV, BASE_FRACTION, STROKE_STRENGTH, PIXEL_WEIGHT,
    CONTROL_CORRECT as EXP013_CONTROL_CORRECT,
)

OUT = ROOT / '05_model_optimize' / 'KNN' / 'Normal' / 'results' / 'EXP015_k_weights_hog'
K_VALUES = (1, 3, 5, 7, 9, 11)
WEIGHTS = ('uniform', 'distance')
EXPECTED_EXP014 = (18851, 2381)  # EXP014 targeted_10 official / Hand-made correct
TARGETED_EXTRA_N = 7680


def predict_from_neighbors(ix: np.ndarray, d: np.ndarray, train_y: np.ndarray,
                           k: int, weights: str) -> np.ndarray:
    """Exact sklearn vote semantics for 16 integer labels, including zero distances.

    Ties are resolved to smallest label (as in sklearn's weighted_mode/mode).
    """
    labels = train_y[ix[:, :k]]
    distances = d[:, :k]
    if weights == 'uniform':
        vote_weights = np.ones_like(distances, dtype=np.float64)
    elif weights == 'distance':
        zero = distances == 0
        vote_weights = np.zeros_like(distances, dtype=np.float64)
        np.divide(1.0, distances, out=vote_weights, where=~zero)
        vote_weights = np.where(zero.any(axis=1, keepdims=True),
                                zero.astype(np.float64), vote_weights)
    else:
        raise ValueError(weights)
    votes = np.zeros((len(labels), len(LABELS)), dtype=np.float64)
    for label in range(len(LABELS)):
        votes[:, label] = np.sum(vote_weights * (labels == label), axis=1)
    return np.argmax(votes, axis=1).astype(np.int64)


def voting_self_test():
    y = np.array([1, 2, 1, 2, 3], dtype=np.int64)
    i = np.array([[0, 1, 2], [1, 3, 4], [0, 1, 4]])
    d = np.array([[1., 2., 3.], [0., 0., 3.], [0., 0., 0.]])
    assert np.array_equal(predict_from_neighbors(i, d, y, 3, 'uniform'), [1, 2, 1])
    assert np.array_equal(predict_from_neighbors(i, d, y, 3, 'distance'), [1, 2, 1])
    print('VOTING SELF-TEST PASSED: uniform, inverse distance, zero-distance ties', flush=True)


def save_predictions(name, pred, yhm, writers, local_indices):
    save_csv(OUT / f'handmade_predictions_{name}.csv', [
        {'writer': int(writers[i]), 'index': int(local_indices[i]),
         'true': LABELS[int(yhm[i])], 'predicted': LABELS[int(pred[i])],
         'correct': int(pred[i] == yhm[i])} for i in range(len(yhm))
    ])


def main():
    test_centering()
    test_augmentation()
    test_deskew()
    test_hog()
    voting_self_test()
    xtr, ytr, xva, yva, sha = fixed_split()
    xhm, yhm, writers, local_indices = read_handmade()
    if (len(ytr), len(yva), len(yhm)) != (76800, 19200, 2560):
        raise RuntimeError('Unexpected data split')
    weak, counts, _ = weak_classes_from_dev_predictions(
        PRED_CSV, yhm, writers, local_indices)
    if [LABELS[i] for i in weak] != ['D', 'B', '5', '3']:
        raise RuntimeError('EXP014 weak-class selection unexpectedly changed')
    print('Selected targeted classes:', [LABELS[i] for i in weak], flush=True)
    print('Official Training=76800; Validation=19200; Hand-made=2560', flush=True)
    OUT.mkdir(parents=True, exist_ok=True)

    # Recreate EXACT EXP014 targeted_10 training features: order, seed, transforms.
    selected = sample_balanced(ytr, fraction=BASE_FRACTION, seed=SEED)
    sized, _ = size_jitter(xtr, selected, seed=SEED)
    strokes, _ = stroke_jitter(sized, strength=STROKE_STRENGTH)
    raw_base = np.concatenate([xtr, strokes], axis=0)
    labels_base = np.concatenate([ytr, ytr[selected]], axis=0)
    del sized, strokes

    com = BatchedCoM()
    t0 = time.perf_counter()
    base_img, _ = deskew(com.transform(raw_base))
    val_img, _ = deskew(com.transform(xva))
    hm_img, _ = deskew(com.transform(xhm))
    del raw_base
    print(f'CoM -> Deskew preprocessing: {time.perf_counter()-t0:.1f}s', flush=True)
    t0 = time.perf_counter()
    base_feat = hybrid_features(base_img, hog_features(base_img), pixel_weight=PIXEL_WEIGHT)
    val_feat = hybrid_features(val_img, hog_features(val_img), pixel_weight=PIXEL_WEIGHT)
    hm_feat = hybrid_features(hm_img, hog_features(hm_img), pixel_weight=PIXEL_WEIGHT)
    del base_img, val_img, hm_img
    print(f'Pixel:HOG 20:80 extraction: {time.perf_counter()-t0:.1f}s', flush=True)

    indices = select_extra(ytr, selected, TARGETED_EXTRA_N, weak, seed=SEED + 142)
    extra_feat, _ = augmented_features(xtr, indices, com, 'targeted_10')
    features = np.ascontiguousarray(np.concatenate([base_feat, extra_feat], axis=0), dtype=np.float32)
    labels = np.concatenate([labels_base, ytr[indices]]).astype(np.int64)
    del base_feat, extra_feat, labels_base
    if len(labels) != 107520 or features.shape != (107520, 1108):
        raise RuntimeError(f'Wrong EXP014 training shape: {features.shape}')
    print(f'EXACT EXP-014 targeted_10 training rows: {len(labels)}, features: {features.shape[1]}', flush=True)

    # sklearn's Euclidean (p=2) brute search; compute all 11 neighbors once.
    knn = KNeighborsClassifier(n_neighbors=max(K_VALUES), weights='distance',
                               metric='euclidean', algorithm='brute', n_jobs=-1)
    t0 = time.perf_counter()
    knn.fit(features, labels)
    fit_sec = time.perf_counter() - t0
    print('Querying 11 neighbors for official Validation (once)...', flush=True)
    t0 = time.perf_counter()
    dv, iv = knn.kneighbors(val_feat, n_neighbors=max(K_VALUES), return_distance=True)
    val_sec = time.perf_counter() - t0
    print(f'Official neighbors obtained in {val_sec:.1f}s', flush=True)
    print('Querying 11 neighbors for Hand-made (once)...', flush=True)
    t0 = time.perf_counter()
    dh, ih = knn.kneighbors(hm_feat, n_neighbors=max(K_VALUES), return_distance=True)
    hm_sec = time.perf_counter() - t0
    print(f'Hand-made neighbors obtained in {hm_sec:.1f}s', flush=True)

    # Check custom vote against sklearn itself on a subset before full comparison.
    for feature_array, indices_array, distances_array in ((val_feat, iv, dv), (hm_feat, ih, dh)):
        knn.set_params(n_neighbors=3, weights='distance')
        sklearn_pred = np.asarray(knn.predict(feature_array[:128]), dtype=np.int64)
        our_pred = predict_from_neighbors(indices_array[:128], distances_array[:128],
                                          labels, 3, 'distance')
        if not np.array_equal(sklearn_pred, our_pred):
            raise RuntimeError('Custom voting does not reproduce sklearn on a test batch')

    rows, writer_rows, class_rows, records = [], [], [], {}
    control_pv = control_ph = None
    for k in K_VALUES:
        for weights in WEIGHTS:
            name = f'k{k}_{weights}'
            pv = predict_from_neighbors(iv, dv, labels, k, weights)
            ph = predict_from_neighbors(ih, dh, labels, k, weights)
            ov, hh = scores(yva, pv), scores(yhm, ph)
            correct = (int(np.sum(pv == yva)), int(np.sum(ph == yhm)))
            if k == 3 and weights == 'distance':
                if correct != EXPECTED_EXP014:
                    raise RuntimeError(
                        f'EXP014 targeted_10 reproduction failed: {correct} != {EXPECTED_EXP014}')
                control_pv, control_ph = pv.copy(), ph.copy()
                print('EXP-014 targeted_10 k3_distance reproduced exactly', flush=True)
            row = {
                'variant': name, 'k': k, 'weights': weights, 'train_n': len(labels),
                'official_correct': correct[0], 'official_accuracy': ov['accuracy'],
                'official_macro_f1': ov['macro_f1'],
                'handmade_correct': correct[1], 'handmade_accuracy': hh['accuracy'],
                'handmade_macro_f1': hh['macro_f1'],
                'official_guardrail_pass': int(ov['accuracy'] >= OFFICIAL_GUARDRAIL),
            }
            rows.append(row)
            records[name] = {'row': row, 'official': ov, 'handmade': hh}
            for w in (41, 42, 43, 44):
                mask = writers == w
                s = scores(yhm[mask], ph[mask])
                writer_rows.append({'variant': name, 'writer': w,
                                    'accuracy': s['accuracy'], 'macro_f1': s['macro_f1']})
            for char in LABELS:
                s = hh['per_class'][char]
                class_rows.append({'variant': name, 'class': char,
                                   'support': s['support'], 'precision': s['precision'],
                                   'recall': s['recall'], 'f1': s['f1']})
            save_csv(OUT / 'candidate_results.csv', rows)
            save_csv(OUT / 'by_writer.csv', writer_rows)
            save_csv(OUT / 'per_class_handmade.csv', class_rows)
            save_cm(OUT / f'official_cm_{name}.csv', ov['confusion_matrix'])
            save_cm(OUT / f'handmade_cm_{name}.csv', hh['confusion_matrix'])
            save_predictions(name, ph, yhm, writers, local_indices)
            (OUT / 'metrics_partial.json').write_text(
                json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
            print(f'{name:<14} Official={ov["accuracy"]:.6f} F1={ov["macro_f1"]:.6f} | '
                  f'Hand-made={hh["accuracy"]:.6f} F1={hh["macro_f1"]:.6f} | '
                  f'guardrail={"PASS" if row["official_guardrail_pass"] else "FAIL"}', flush=True)

    if control_pv is None:
        raise RuntimeError('Missing control predictions')
    for name, rec in records.items():
        if name == 'k3_distance':
            continue
        # Predictions can be reconstructed from saved nearest neighbor arrays here.
        k, weight = rec['row']['k'], rec['row']['weights']
        pv = predict_from_neighbors(iv, dv, labels, k, weight)
        ph = predict_from_neighbors(ih, dh, labels, k, weight)
        rec['paired_vs_exp014'] = {
            'official': paired_stats(yva, control_pv, pv),
            'handmade': paired_stats(yhm, control_ph, ph),
        }
    eligible = [r for r in rows if r['official_guardrail_pass']]
    if not eligible:
        raise RuntimeError('No eligible candidate')
    best = max(eligible, key=lambda r: (r['handmade_macro_f1'],
                                       r['handmade_accuracy'], r['official_accuracy']))
    summary = {
        'experiment': 'EXP-015', 'dataset_sha256': sha,
        'official_train_n': len(ytr), 'official_validation_n': len(yva),
        'handmade_development_n': len(yhm),
        'knn': {'metric': 'euclidean', 'k_values': list(K_VALUES), 'weights': list(WEIGHTS)},
        'preprocessing': 'CoM -> Deskew', 'pixel_weight': PIXEL_WEIGHT,
        'baseline_augmentation_fraction': BASE_FRACTION,
        'stroke_strength': STROKE_STRENGTH,
        'targeted_classes': [LABELS[c] for c in weak],
        'targeted_extra_n': TARGETED_EXTRA_N,
        'training_n': len(labels), 'feature_dimensions': features.shape[1],
        'neighbor_query_seconds': {'fit': fit_sec, 'official': val_sec, 'handmade': hm_sec},
        'candidates': rows, 'selected_development_candidate': best,
        'results': records,
        'environment': {'python': platform.python_version(),
                        'numpy': np.__version__, 'sklearn': sklearn.__version__},
        'limitations': 'Hand-made development labels have already been used to choose '
                       'targeted classes and tune preprocessing. No independent Test used.',
    }
    (OUT / 'metrics.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print('\nSELECTED DEVELOPMENT CANDIDATE:', best['variant'], flush=True)
    print(f'Official={best["official_accuracy"]:.6f}, Hand-made={best["handmade_accuracy"]:.6f}, '
          f'MacroF1={best["handmade_macro_f1"]:.6f}', flush=True)
    print('Saved:', OUT, flush=True)


if __name__ == '__main__':
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")